# Schema design

Constraints keep invalid relationships and duplicate values out of the database.

In [ ]:
# Constraints protect data even when writes come from another code path.
schema = """
CREATE TABLE users (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    email TEXT NOT NULL UNIQUE
);

CREATE TABLE tasks (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    owner_id BIGINT NOT NULL REFERENCES users(id),
    title TEXT NOT NULL
);
"""

print(schema)

## Polished version

A standalone schema enforces identity, uniqueness, ownership, valid values, and cascading deletes in the database.

In [ ]:
# Build and inspect the schema in memory so the example stays standalone.
import sqlite3


def create_schema(connection: sqlite3.Connection) -> None:
    # SQLite requires foreign-key enforcement to be enabled per connection.
    connection.execute("PRAGMA foreign_keys = ON")
    connection.executescript(
        """
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL UNIQUE
        );

        CREATE TABLE tasks (
            id INTEGER PRIMARY KEY,
            owner_id INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
            title TEXT NOT NULL CHECK (length(title) BETWEEN 1 AND 200),
            completed INTEGER NOT NULL DEFAULT 0 CHECK (completed IN (0, 1)),
            created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
        );

        CREATE INDEX ix_tasks_owner_id ON tasks(owner_id);
        """
    )


connection = sqlite3.connect(":memory:")
create_schema(connection)
# Parameterized writes keep values separate from SQL syntax.
connection.execute("INSERT INTO users (email) VALUES (?)", ("ada@example.com",))
connection.execute(
    "INSERT INTO tasks (owner_id, title) VALUES (?, ?)",
    (1, "Design constraints"),
)

columns = connection.execute("PRAGMA table_info(tasks)").fetchall()
indexes = connection.execute("PRAGMA index_list(tasks)").fetchall()
print("Columns:", [column[1] for column in columns])
print("Indexes:", [index[1] for index in indexes])

## Applied in this repository

The production schema is declared in [models.py](../00P1-project-rest-api/app/models.py) and created reversibly by [the Alembic migration](../00P1-project-rest-api/migrations/versions/0001_create_users_and_tasks.py).